# DataMate · Colab inference đúng 200 câu benchmark nhóm

Mẫu cố định: **seed 4000930841**, 200 câu TRAIN, 28 doanh nghiệp. Notebook này
chạy agent hiện có của bạn; không thay cách chấm hay yêu cầu của các đồng đội.

1. Upload notebook này lên Colab. Đưa **datamate-team200-colab-kit.zip** vào
   **MyDrive/DataMate/**. Model đã seal ở MyDrive/DataMate/models/ được dùng lại.
2. Chọn GPU G4, sửa tên/model/đường dẫn ở ô đầu, rồi **Run all**.
3. Giữ RUN_LIMIT=None để chạy đủ 200; AUTO_DISCONNECT=True để lưu ZIP,
   đồng bộ Drive rồi tự ngắt runtime. Mặc định Qwen 9B/self_review như pipeline
   6 đã khôi phục; có thể chọn Gemma 12B hoặc checkpoint đã hỗ trợ ở ô đầu.
   Preflight phải hiện pipeline 6/backend 3. Dùng tên lượt layout-Nov14th mới,
   không resume vào kết quả v7 hoặc lượt v6 đã chạy trước đó.

Kết quả ở MyDrive/DataMate/runs/team200/<RUN_NAME>/:
- **group_submission.zip**: chỉ submission.jsonl theo schema cuộc thi và
  run_metadata.json sáu trường. Đây là ZIP gửi nhóm.
- **diagnostic_bundle.zip**: review.csv/jsonl/md, summary/runtime/manifest,
  prompt/raw output/SQL/tool trong logs/*.json và logs/*.md. Có nhãn chuẩn,
  giữ riêng để phân tích sau khi khóa kết quả.

PA trên review có mẫu số 200 kể cả nếu dừng sớm. Nhãn chỉ được đọc sau vòng
inference để review, không đưa cho agent. Thư viện/model chỉ tải ở bước chuẩn
bị; inference dùng checkpoint local và chế độ offline. Đây là benchmark nội
bộ TRAIN, không phải holdout độc lập.

Resume: giữ cùng RUN_NAME, model, code, config và loại GPU/runtime. Nếu vẫn
cùng kernel chỉ chạy lại ô batch; kernel mới thì Run all. Log đã ghi được bỏ
qua, không chọn/ghép đáp án. Khác nguồn/cấu hình/runtime bị chặn. Nếu máy bị
tắt đột ngột không đo đủ thời gian, metadata giữ timing null thay vì bịa số.
Unit giữ null khi chưa đo thực tế; không dùng 10 unit/giờ làm unit đã tiêu.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import hashlib, importlib, json, os, runpy, shutil, stat, subprocess, sys, time, uuid, zipfile

BASE = Path("/content/drive/MyDrive/DataMate")  # Sửa nếu Drive của bạn dùng thư mục khác.
KIT_ZIP = BASE / "datamate-team200-colab-kit.zip"
PARTICIPANT = "toi"                # Tên dùng trong file nộp nhóm.
MODEL_KEY = "qwen35-9b"             # Hoặc "gemma4-12b", "qwen35-upscaled14b-v2".
QUANTIZATION = "none"              # G4 96 GB: none; GPU ít VRAM có thể chọn 4bit.
MODE = "self_review"               # Giữ một cấu hình cho cả lượt chạy.
BACKEND = "torch"                 # torch | vllm; chọn độc lập với RAG.
RAG_PRESET = "rag-none"            # Baseline trước; rag-dense hoặc rag-dense-rerank sau.
RUN_NAME = f"{PARTICIPANT}-{MODEL_KEY}-{BACKEND}-{RAG_PRESET}-team200-layout-Nov14th-01"
AUTO_DISCONNECT = True
RUN_LIMIT = None                    # None = đủ 200; chỉ đặt 5 nếu muốn chạy thử rồi resume.
MAX_RUN_MINUTES = 180               # Ngân sách MỖI phiên batch, không gồm chuẩn bị/load.
COMPUTE_UNITS_USED = None           # Chỉ điền số đo thật; không suy từ 10 unit/giờ.
SAVE = BASE / "runs/team200" / RUN_NAME
MODEL_SOURCE_DIR = BASE / "models" / MODEL_KEY
if AUTO_DISCONNECT and not SAVE.resolve().is_relative_to(Path("/content/drive").resolve()):
    raise RuntimeError("AUTO_DISCONNECT cần lưu kết quả trên Drive; sửa BASE hoặc tắt tự ngắt.")

EXPECTED = {
  "datamate-team200-source.zip": "930b2bd31803e80b44ff0cdadf2ffff1f9ba6ca496967ee5cdf0419284ce7d21",
  "participants.zip": "f24e938bd60764034c9272881aa760b53ebc0116a8975e3449678d25a7151652",
  "coordinator-private.zip": "0bf2b5c1f44ea56f63ea77d61cb64aa0d53cb7006761d524d0bdadb9e092a3a3"
}
if any(name == "datamate" or name.startswith("datamate.") for name in sys.modules):
    raise RuntimeError("Kernel đã import DataMate. Restart session/kernel rồi chạy từ ô đầu; giữ cache /content/models. Muốn resume cùng agent chỉ chạy lại ô batch cuối.")
if not KIT_ZIP.is_file():
    raise FileNotFoundError(f"Đưa datamate-team200-colab-kit.zip lên Drive tại {KIT_ZIP}")

def digest(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def extract_checked(archive_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        if archive.testzip() is not None:
            raise ValueError("ZIP lỗi CRC")
        seen = set()
        for info in archive.infolist():
            name = info.filename.replace("\\", "/")
            target = (destination / name).resolve()
            if (name in seen or not target.is_relative_to(destination)
                    or ":" in name or stat.S_ISLNK(info.external_attr >> 16)):
                raise ValueError(f"ZIP path không hợp lệ: {name}")
            seen.add(name)
            if not info.is_dir():
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)

RUNTIME_ROOT = Path("/content/datamate-team200")
INPUTS = RUNTIME_ROOT / "inputs"
INPUTS.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(KIT_ZIP) as archive:
    for name, expected in EXPECTED.items():
        target = INPUTS / name
        with archive.open(name) as source, target.open("wb") as output:
            shutil.copyfileobj(source, output)
        if digest(target) != expected:
            raise ValueError(f"Gói sai phiên bản/khác hash: {name}")
SOURCE_SHA256 = EXPECTED["datamate-team200-source.zip"]
WORK = RUNTIME_ROOT / ("source-" + SOURCE_SHA256[:12])
extract_checked(INPUTS / "datamate-team200-source.zip", WORK)
BENCHMARK_FOLDER = RUNTIME_ROOT / "benchmark-v2"
extract_checked(INPUTS / "participants.zip", BENCHMARK_FOLDER)
PARTICIPANT_ROOT = BENCHMARK_FOLDER / "participant"
TRAIN_ROOT = PARTICIPANT_ROOT / "dataset"  # Chỉ là tên biến; không có answers.jsonl.
REFERENCE_ZIP = INPUTS / "coordinator-private.zip"  # Chỉ mở sau vòng inference để review.
check = runpy.run_path(str(PARTICIPANT_ROOT / "benchmark.py"))
benchmark, questions = check["verify"](PARTICIPANT_ROOT)
assert benchmark["benchmark_id"] == "datamate-train200-seed4000930841"
assert benchmark["questions_sha256"] == "7ae2079219d0a79ca1dd66952610ceb091e36c0ecf65359b11dac8263c67b782"
assert len(questions) == 200 and not (TRAIN_ROOT / "answers.jsonl").exists()
os.chdir(WORK)
sys.path.insert(0, str(WORK))
sys.path.insert(0, str(WORK / "src"))
os.environ.update(HF_HOME=str(WORK / ".cache/huggingface"), TORCH_HOME=str(WORK / ".cache/torch"),
                  PIP_CACHE_DIR=str(WORK / ".cache/pip"), TOKENIZERS_PARALLELISM="false")
print("Đúng 200 câu, seed 4000930841; không bốc lại mẫu.")
print("Model:", MODEL_KEY, "| mode:", MODE, "| kết quả:", SAVE)


## Cài dependency và kiểm tra GPU
Giữ PyTorch CUDA có sẵn của Colab; preflight chạy trước khi nạp trọng số.


In [ ]:

if RAG_PRESET not in {"rag-none", "rag-dense", "rag-dense-rerank"}:
    raise ValueError("RAG_PRESET must be rag-none, rag-dense, or rag-dense-rerank")
RAG_MODE = {"rag-none":"none","rag-dense":"dense","rag-dense-rerank":"dense_rerank"}[RAG_PRESET]
CONFIG_FILE = WORK / "configs" / "experiments" / "thinh" / (RAG_PRESET + ".json")
import importlib
from importlib.metadata import version, PackageNotFoundError
source_dir = (WORK / "src").resolve()
if not (source_dir / "datamate/__init__.py").is_file():
    raise RuntimeError("Source ZIP lacks src/datamate; upload a complete package")
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))
importlib.invalidate_caches()
from datamate.config import Settings
from datamate.models.registry import inference_extras
overrides = {"backend":globals().get("BACKEND")}
if globals().get("RAG_MODE") is not None:
    overrides["rag"] = {"mode":RAG_MODE}
selected = Settings.load(globals().get("CONFIG_FILE"), **overrides)
extras = inference_extras(selected)
def installed(name):
    try: return version(name)
    except PackageNotFoundError: return None
loaded_versions = {name:installed(name) for name in ("torch","transformers","tokenizers","numpy","vllm") if name in sys.modules}
install = [sys.executable,"-m","pip","install","-e",".[" + extras + "]"]
if selected.backend != "vllm":
    import torch
    constraints = WORK / "colab-constraints.txt"
    constraints.write_text("torch==" + torch.__version__ + "\n", encoding="utf-8")
    install += ["-c",str(constraints)]
subprocess.run(install,cwd=WORK,check=True)
changed = [name for name,before in loaded_versions.items() if installed(name) != before]
if changed:
    raise RuntimeError("Packages changed while loaded: " + str(changed) + "; restart kernel and rerun before inference")
if globals().get("CHECK_CUDA",True):
    import torch
    subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv"],check=True)
    assert torch.cuda.is_available(), "Select a CUDA GPU runtime first"
    print("PyTorch:",torch.__version__,"CUDA:",torch.version.cuda)
import datamate
print("DataMate import OK:", datamate.__version__, datamate.__file__, "extras:",extras)

verify_runtime = runpy.run_path(str(WORK / "tools/colab_runtime_check.py"))["verify_runtime"]
print("Preflight:", verify_runtime(WORK))


## Chuẩn bị checkpoint và nạp model
Checkpoint/revision được stage-model kiểm tra và seal; không tự dùng main.


In [ ]:
from datamate.config import Settings
from datamate.application import create_agent
verify_runtime(WORK)
CONFIG_PATH = WORK / "configs" / "experiments" / "thinh" / f"{RAG_PRESET}.json"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing RAG preset: {CONFIG_PATH}")

def ensure_staged(key, directory):
    directory = Path(directory)
    if (directory / "artifact_manifest.json").is_file():
        print("Checkpoint already sealed:", key, directory, flush=True)
        return directory
    stage_env = {**os.environ, "HF_HUB_OFFLINE": "0", "TRANSFORMERS_OFFLINE": "0"}
    subprocess.run([sys.executable, "-m", "datamate", "stage-model", "--model", key,
                    "--output", str(directory)], cwd=WORK, env=stage_env, check=True)
    return directory

if "agent" in globals():
    raise RuntimeError("Model đã load. Chạy lại ô batch để resume; không load chồng hai model.")
# Đây là bước chuẩn bị online, tách khỏi thời gian load/inference benchmark.
ensure_staged(MODEL_KEY, MODEL_SOURCE_DIR)
LOAD_SESSION_ID = uuid.uuid4().hex
load_started = time.monotonic()
prepare_local_model = runpy.run_path(str(WORK / "tools/colab_model_cache.py"))["prepare_local_model"]
MODEL_DIR = prepare_local_model(MODEL_SOURCE_DIR, Path("/content/models"), MODEL_KEY)
settings = Settings.load(CONFIG_PATH, use_env=False, model_key=MODEL_KEY, model_dir=str(MODEL_DIR),
                         backend=BACKEND,
                         mode=MODE, quantization=QUANTIZATION, split_root=str(TRAIN_ROOT),
                         cache_dir=str(WORK / ".cache/datamate-team200" / RAG_PRESET), result_dir=str(SAVE))
if settings.rag.mode != "none":
    settings.rag.device = "cuda:0"
    for role in ["embedding"] + (["reranker"] if settings.rag.mode == "dense_rerank" else []):
        key = getattr(settings.rag, role + "_key")
        sealed = ensure_staged(key, BASE / "models" / key)
        cached = prepare_local_model(sealed, Path("/content/models"), key)
        setattr(settings.rag, role + "_dir", str(cached))
settings.validate()
agent = create_agent(settings)
LOAD_SECONDS = round(time.monotonic() - load_started, 4)
print("Loaded:", MODEL_KEY, "| pipeline:", agent.pipeline_version, "| load:", LOAD_SECONDS, "giây")
print("Generation tokens:", getattr(agent.backend, "generation_tokens", None))


## Chạy 200 câu, lưu review/log và tự ngắt
ZIP được kiểm tra nội dung rồi đồng bộ Drive trước khi kết thúc runtime. Lượt partial cũng được lưu để resume.


In [ ]:
runner = runpy.run_path(str(WORK / "tools/infer_team200.py"))
# Biên nhận riêng cho lần này, không dùng lại ZIP cũ nếu gặp lỗi trước khi lưu.
INVOCATION_ID = uuid.uuid4().hex
try:
    summary, group_zip, diagnostic_zip = runner["run_team200"](
        agent, PARTICIPANT_ROOT, SAVE, REFERENCE_ZIP,
        participant=PARTICIPANT, load_seconds=LOAD_SECONDS, load_session_id=LOAD_SESSION_ID,
        source_archive_sha256=SOURCE_SHA256, limit=RUN_LIMIT,
        wall_budget_seconds=MAX_RUN_MINUTES * 60, units_used=COMPUTE_UNITS_USED,
        invocation_id=INVOCATION_ID,
    )
finally:
    receipt_path = SAVE / "report_receipt.json"
    receipt = json.loads(receipt_path.read_text(encoding="utf-8")) if receipt_path.is_file() else {}
    if receipt.get("invocation_id") == INVOCATION_ID:
        print("Đã lưu hai ZIP trên Drive:", SAVE, flush=True)
        runner["finish_and_disconnect"](SAVE, enabled=AUTO_DISCONNECT)
    else:
        print("Chưa xác nhận có báo cáo mới đầy đủ; giữ runtime để bạn kiểm tra lỗi.", flush=True)
